In this Project we will learn about all all advanced topic of LangGraph


In [1]:
from langgraph.graph import StateGraph , END , START
from typing import TypedDict , Literal , Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage , HumanMessage
import operator
from langgraph.checkpoint.memory import MemorySaver ## it is form persistance

In [2]:
# Define state :

from langgraph.graph import add_messages # this is optimized method for better use with BaseMessages

class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage] , add_messages] # BaseMessage consists all messages(human , system , tool etc)

In [3]:
# task :

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

def chat_node(state: ChatState):

    # take use query form state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state : 
    return {
        'messages' : [response]
    }

In [4]:
# Graph : 

checkpointer = MemorySaver()

graph = StateGraph(ChatState)

# add nodes :

graph.add_node('chat_node' , chat_node)

# add edges :

graph.add_edge(START , "chat_node")
graph.add_edge('chat_node' , END)

# compile : 

chatbot = graph.compile(checkpointer=checkpointer)


In [9]:
from langchain_core.messages import HumanMessage

initial_state = {
    "messages": [
        HumanMessage(content="What is AI?")
    ]
}

config = {
    "configurable": {
        "thread_id": "user_1"
    }
}

response = chatbot.invoke(
    initial_state,
    config=config
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': '**Artificial Intelligence (AI)** refers to the development of computer systems capable of performing tasks that traditionally require human intelligence. These tasks include learning from experience, recognizing patterns, understanding natural language, solving complex problems, and making decisions.\n\nAt its core, AI is about creating software that can "think," "reason," and "learn" from data.\n\n---\n\n### How Does AI Work?\nAI doesn\'t think like a biological human brain, but it mimics cognitive functions using **data, algorithms, and computing power**. \n\n1. **Data Collection:** AI systems require vast amounts of data (text, images, numbers, audio) to understand patterns.\n2. **Algorithms:** An algorithm is a set of rules or instructions given to the AI.\n3. **Training & Learning:** The AI processes the data using these rules, makes predictions, checks if it was right or wrong, and adjusts itself to get better next time.\n\n#### Key Subfields of AI:\n* 

In [8]:
thread_id = '1'


while True:
        user_message = input("\n🧑 You: ").strip()

        # Ignore empty messages
        if not user_message:
            continue

        print('User :' , user_message)

        # Exit condition
        if user_message.lower() in {"exit", "quit", "bye"}:
            print("\n👋 Goodbye! Have a great day.")
            break

        print("\n🤖 AI is thinking...\n")

        config = {"configurable" : {'thread_id' : thread_id}}

        response = chatbot.invoke(
            {"messages": [HumanMessage(content=user_message)]},
            config= config
        )

        print(f"🤖 AI: {response['messages'][-1].text}")



User : hlw

🤖 AI is thinking...

🤖 AI: Hello! How can I help you today?
User : by

🤖 AI is thinking...

🤖 AI: Goodbye! Have a great day, and feel free to reach out whenever you need help!
User : by

🤖 AI is thinking...

🤖 AI: Bye! Take care! 👋
User : exit

👋 Goodbye! Have a great day.


In [ ]:
chatbot.get_state(config = config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is ayush', additional_kwargs={}, response_metadata={}, id='09f8b938-9552-4f6b-8a97-cfd025f6daff'), AIMessage(content=[{'type': 'text', 'text': "Hello Ayush! It's great to meet you. How can I help you today?", 'extras': {'signature': 'Eo8HCowHARFNMg/jS5goG36cjSU3nNbmpQDIq1yLqzu0scVf3MKawivTvB+xCdkXLoy/hZRQf6CPUKYUGxoGbFodcgWC71pIZflv93rdnoHjSDXYb+IXcl1wa6PLdr9k6ZDWi4bMS4O7K5Pjk8GFJnvirxS0JeNr85B47vqXqiuWpmlMWvw1/4SnOsvHEvFMkDK24BtHrqn4dKvXHuooUwVCNuO5xdVU3secLXIDQLv8wdIyu7yvVBGcsuJ9zkhnq5uusUFKLD4QnVlpQ9x+5k7s2mJaqFOwmFL+7bzvRz426xuKeK26oR/LmIPmfPwBwPMoG9FIh5ipNvU6J5YZ5SZRQCW+4nXwWYzmYWDtcyQBWXFQuCGfo3dTd9JOc+rSls6KPw74DbtWmu+ez4Z9QrE6bpX7FhcNJQgxd8omyi3nfB26NJ/tgnk2U29W7G0MceFeUjaO1hV1Ei95Xs1M0098/4zxkF/Ji/0MvxRsJJTq5fddxfnVVzZhBLELm+NHgnIUIaigPE7iKtz5rHLDt3Mk7sb4+4cdIAlK4jocXvvp+fwL7gec1+2++qwJDM0qTvNVMR7MTOc9+GLI6gsKmGnuVYj01Bv9xyl7GoqBC3dPgxYnMD4n6SA0GMfgW9cElCEZFRgOK3C4GtWIf3/8Xp+MbajEGu8zmSmqFGT2uOYuAzzbG/fEpa5MMi